<!-- track-identity-card -->
# Normalization against the reinforcement-learning reference

| | |
|---|---|
| Pipeline step | `08_reference_gap.ipynb` |
| Manuscript section | 4.5 |
| Copied from | `notebooks/NB15_sac_gap_analysis_v3_2_2026-07-22_2150.ipynb` |
| Source sha256 | `f675abd180447779c96529eb62b74b83` |

**Reads**

- `data/features/driver_corner_matrix*<track>*.parquet`
- `data/processed/tier3_large_3t3c/*`

**Writes**

- `results/gap_analysis_<policy>/nb15_info.json`
- `results/gap_analysis_<policy>/sac_reference_matrix.parquet`

Produces the agent reference and the ratio classification of Section 4.5.

> Copied from the working notebook named above. Two changes were made to it: this identity card and the bootstrap cell that follows it were added, and the hard-coded data paths were replaced with the root that the bootstrap cell resolves. The analysis code is unchanged.


# NB15 v2 - SAC Gap Analizi / SAC-ratio Normalizasyon (HNS inversiyonu)

Amac: SAC (reward-optimal ajan) referansini insan/Efe metrikleriyle ayni metrik uzayinda uretmek, sonra `insan_metrik / SAC_metrik` orani ile normalize etmek (Mnih 2015 Human Normalized Score'un tersi).

**Tasarim:** Temiz SAC turlari -> corners_v3 + segment_corner (NB14 ile AYNI fonksiyonlar) -> viraj metrikleri -> agregat SAC referansi. SAC referansi ~0 olan metrikler degenerate kabul edilip Min-Max'ta kalir; anlamli olanlar SAC-oran alir. Pistler: Monza, Barcelona, RBR.

In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  KOSU AYARI — SADECE BURAYI DEGISTIR             ║
# ╚══════════════════════════════════════════════════╝
POLICY = "P_EXC"   # "P_INC" = A-blogu DAHIL | "P_EXC" = A-blogu HARIC

# --- ORTAK KIMLIK POLITIKASI (NB11 v3 / NB7A7 v2 ile AYNI liste) ---
EXCLUDE_HARD = [
    "20240213_000000_ACAI",   # oyun-ici yapay surucu
    "20240308_ensemble",      # RL politika ciktisi
    "20240501_MPC",           # klasik kontrol baseline (veri kumesi belgesi)
]
ABLOCK = [
    "20240410_A_12_123",
    "20240410_A_21_231",
    "20240411_A_12_312",
]
assert POLICY in ("P_INC", "P_EXC"), "POLICY 'P_INC' ya da 'P_EXC' olmali"
EXCLUDE_IDS = EXCLUDE_HARD + ([] if POLICY == "P_INC" else ABLOCK)
POL_SUF = "_inc" if POLICY == "P_INC" else "_exc"

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

PROJECT      = TRACK_ROOT
FEATURES_DIR = PROJECT / "data" / "features"
FP_DIR       = PROJECT / "data" / "fingerprints"
SAC_DIR      = PROJECT / "data" / "reference_sac"
# v3: politika ekli cikti — iki politika birbirini EZMEZ, eski arsiv korunur
OUT_DIR      = PROJECT / "results" / ("gap_analysis" + POL_SUF)
OUT_DIR.mkdir(parents=True, exist_ok=True)

_HUMAN_LOG = {}
def read_human(path):
    """Insan matrisini okur ve ORTAK kimlik politikasini uygular.

    v2'de bu filtre YOKTU: features/ koku okunuyordu (Barcelona 24 = ACAI dahil),
    kademe hatti ise features/t3 (23) kullaniyordu. Populasyon celiskisi buydu;
    artik tek kural her iki hatta da uygulaniyor.
    """
    _df = pd.read_parquet(path)
    _n0 = int(_df["driver_id"].nunique())
    _hit = sorted(set(_df["driver_id"]) & set(EXCLUDE_IDS))
    _df = _df[~_df["driver_id"].isin(EXCLUDE_IDS)].copy()
    _n1 = int(_df["driver_id"].nunique())
    _HUMAN_LOG[Path(path).name] = {"before": _n0, "after": _n1, "removed": _hit}
    print(f"  [kimlik filtresi] {Path(path).name}: {_n0} -> {_n1}  cikarilan: {_hit or '-'}")
    # v3.1 ZORUNLU: filtre indekste BOSLUK birakir. Asagida norm/fp cerceveleri
    # temiz RangeIndex ile kuruluyor; bosluklu indeksten indeks-hizali atama
    # satirlari SESSIZCE dusurur (v3'te Monza EXC: 14 surucu -> 11 gecerli).
    return _df.reset_index(drop=True)

N_SAC_PARQUET = len(list(SAC_DIR.rglob("*.parquet")))
print(f"Kimlik politikasi: {POLICY}  ->  {len(EXCLUDE_IDS)} kimlik dislaniyor")

# Segmentasyon parametreleri (NB08 v4 / NB14 ile AYNI)
SEARCH_BACK_M      = 300
SEARCH_FWD_M       = 200
TRAIL_BRAKE_ZONE   = 0.40
TRAIL_MIN_PRESSURE = 0.03
MIN_APEX_SPEED     = 20.0
MIN_ENTRY_SPEED    = 50.0
MC_RADIUS_M        = 15
EXIT_FALLBACK_M    = 50

# SAC temizlik / orneklem
SAC_SAMPLE_N     = 150
MIN_LAP_SPAN_M   = 5000.0
MIN_LAP_MAXSPEED = 150.0
RATIO_EPS        = 1e-3
SAC_READ_COLS    = ['speed','brakeStatus','accStatus','LapDist','steerAngle','going_backwards','isInPit']

# 19-metrik boyut haritasi (NB09 v3 / NB14 ile AYNI)
DIMENSIONS = {
    'B1_Hiz': {'label':'Hiz Yonetimi',
        'metrics':['mean_apex_speed','speed_loss_eff','mean_mc_speed_ratio','mean_mc_lateral','mean_cex_accel_rate']},
    'B2_Frenleme': {'label':'Frenleme Stili',
        'metrics':['mean_brake_pressure','trail_braking_ratio','mean_trail_pressure','mean_slb_dist','mean_slb_decel','mean_ce_brake_turnin']},
    'B3_Strateji': {'label':'Surus Stratejisi',
        'metrics':['mean_coasting_dist','mean_cex_throttle_lag','pct_lift_coast','pct_flat_out']},
    'B4_Tutarlilik': {'label':'Tutarlilik',
        'metrics':['apex_speed_std','exit_speed_std','speed_loss_eff_std','braking_dist_std']},
}
DIM_NAMES = list(DIMENSIONS.keys())
ALL_METRICS = []
for cfg in DIMENSIONS.values():
    ALL_METRICS.extend(cfg['metrics'])

print(f"Config OK - {len(ALL_METRICS)} metrik, {len(DIM_NAMES)} boyut")
print(f"SAC_DIR var mi: {SAC_DIR.exists()} ({len(list(SAC_DIR.rglob('*.parquet')))} parquet)")
print(f"OUT_DIR: {OUT_DIR}")

## 1. Cekirdek Fonksiyonlar (NB14'ten birebir)

In [ ]:
def _empty_seg(apex_lapdist, apex_speed=np.nan):
    keys = [
        'seg_entry_dist', 'seg_apex_dist', 'seg_exit_dist',
        'seg_entry_speed', 'seg_apex_speed', 'seg_exit_speed',
        'seg_exit_method',
        'seg_braking_dist', 'seg_trail_braking', 'seg_trail_pressure',
        'seg_avg_brake_pressure', 'seg_coasting_dist',
        'seg_speed_loss_eff', 'seg_corner_width',
        'phase_slb_dist', 'phase_slb_decel_rate',
        'phase_ce_dist', 'phase_ce_brake_at_turnin',
        'phase_mc_speed_ratio', 'phase_mc_lateral_signal',
        'phase_cex_throttle_lag', 'phase_cex_accel_rate',
        'phase_turnin_method',
    ]
    result = {k: np.nan for k in keys}
    result['seg_apex_dist'] = float(apex_lapdist)
    result['seg_apex_speed'] = float(apex_speed) if not np.isnan(apex_speed) else np.nan
    result['seg_exit_method'] = 'skipped'
    result['seg_entry_valid'] = False
    result['seg_trail_braking'] = False
    return result


def votes_to_confidence(n):
    return {0: 0.0, 1: 0.3, 2: 0.6, 3: 0.85, 4: 1.0}.get(int(n), 0.5)


def segment_corner(df_lap, apex_lapdist, difficulty):
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values   if 'accStatus'   in df_lap.columns else np.zeros(len(df_lap))
    dist  = df_lap['LapDist'].values

    apex_idx = int(np.argmin(np.abs(dist - apex_lapdist)))
    apex_speed = float(speed[apex_idx])

    back_limit = max(0, apex_idx - int(SEARCH_BACK_M / 2))
    entry_idx  = apex_idx
    for j in range(apex_idx - 1, back_limit, -1):
        if brake[j] < 0.05 and speed[j] > speed[apex_idx]:
            entry_idx = j
            break

    entry_speed = float(speed[entry_idx])
    if entry_speed < MIN_ENTRY_SPEED or apex_speed < MIN_APEX_SPEED:
        return _empty_seg(apex_lapdist, apex_speed)

    # EXIT: 3-katmanli tespit (v4)
    fwd_limit = min(len(speed) - 1, apex_idx + int(SEARCH_FWD_M / 2))
    exit_idx    = apex_idx
    exit_method = 'none'

    for j in range(apex_idx + 1, fwd_limit):
        if acc[j] > 0.3 and speed[j] > apex_speed:
            exit_idx = j
            exit_method = 'throttle_full'
            break

    if exit_idx == apex_idx:
        for j in range(apex_idx + 1, fwd_limit):
            if acc[j] > 0.3:
                exit_idx = j
                exit_method = 'throttle_only'
                break

    if exit_idx == apex_idx:
        fb_target = dist[apex_idx] + EXIT_FALLBACK_M
        fb_idx = int(np.argmin(np.abs(dist - fb_target)))
        exit_idx = min(fb_idx, fwd_limit)
        exit_method = 'distance_fallback'

    exit_speed = float(speed[exit_idx])

    entry_dist_v = float(dist[entry_idx])
    exit_dist_v  = float(dist[exit_idx])
    braking_dist = float(dist[apex_idx] - dist[entry_idx]) if entry_idx < apex_idx else 0.0
    speed_loss_eff = (apex_speed / entry_speed) if entry_speed > 0 else 0.0

    coast_start = apex_idx
    for j in range(apex_idx, fwd_limit):
        if brake[j] < 0.05:
            coast_start = j
            break
    coast_end = coast_start
    for j in range(coast_start, fwd_limit):
        if acc[j] > 0.1:
            coast_end = j
            break
    coasting_dist = float(dist[coast_end] - dist[coast_start]) if coast_end > coast_start else 0.0

    if braking_dist > 10:
        trail_zone_start = entry_idx + int((apex_idx - entry_idx) * (1 - TRAIL_BRAKE_ZONE))
        trail_pressures  = brake[trail_zone_start:apex_idx]
        trail_active     = trail_pressures.mean() > TRAIL_MIN_PRESSURE if len(trail_pressures) > 0 else False
        trail_pressure   = float(trail_pressures.mean()) if len(trail_pressures) > 0 else 0.0
    else:
        trail_active   = False
        trail_pressure = 0.0

    avg_brake = float(brake[entry_idx:apex_idx].mean()) if apex_idx > entry_idx else 0.0

    return {
        'seg_entry_dist': entry_dist_v, 'seg_apex_dist': float(dist[apex_idx]),
        'seg_exit_dist': exit_dist_v,
        'seg_entry_speed': entry_speed, 'seg_apex_speed': apex_speed,
        'seg_exit_speed': exit_speed,
        'seg_exit_method': exit_method,
        'seg_braking_dist': braking_dist, 'seg_trail_braking': trail_active,
        'seg_trail_pressure': trail_pressure, 'seg_avg_brake_pressure': avg_brake,
        'seg_coasting_dist': coasting_dist, 'seg_speed_loss_eff': speed_loss_eff,
        'seg_corner_width': exit_dist_v - entry_dist_v,
        'seg_entry_valid': True,
    }


def segment_corner_phases(df_lap, seg, steer_info):
    mode, steer_col = steer_info

    if not seg.get('seg_entry_valid', False) or pd.isna(seg.get('seg_entry_dist')):
        return {
            'phase_slb_dist': np.nan, 'phase_slb_decel_rate': np.nan,
            'phase_ce_dist': np.nan, 'phase_ce_brake_at_turnin': np.nan,
            'phase_mc_speed_ratio': np.nan, 'phase_mc_lateral_signal': np.nan,
            'phase_cex_throttle_lag': np.nan, 'phase_cex_accel_rate': np.nan,
            'phase_turnin_method': 'skipped',
        }

    dist  = df_lap['LapDist'].values
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values if 'accStatus' in df_lap.columns else np.zeros(len(df_lap))

    entry_dist = seg['seg_entry_dist']
    apex_dist  = seg['seg_apex_dist']
    exit_dist  = seg['seg_exit_dist']

    entry_idx = int(np.argmin(np.abs(dist - entry_dist)))
    apex_idx  = int(np.argmin(np.abs(dist - apex_dist)))
    exit_idx  = int(np.argmin(np.abs(dist - exit_dist)))

    # Turn-in noktasi tespiti
    turnin_idx = entry_idx + int((apex_idx - entry_idx) * 0.4)
    turnin_method = 'speed_proxy_40pct'

    if mode == 'steer' and steer_col in df_lap.columns:
        steer = np.abs(df_lap[steer_col].values)
        region = steer[entry_idx:apex_idx]
        if len(region) > 5:
            wlen = min(11, len(region))
            if wlen % 2 == 0:
                wlen -= 1
            if wlen >= 3:
                smoothed = savgol_filter(region, wlen, min(2, wlen - 1))
            else:
                smoothed = region
            threshold = smoothed.max() * 0.15
            for k, val in enumerate(smoothed):
                if val > threshold:
                    turnin_idx = entry_idx + k
                    turnin_method = 'steer_threshold'
                    break

    elif mode == 'g_lat' and steer_col in df_lap.columns:
        glat = np.abs(df_lap[steer_col].values)
        region = glat[entry_idx:apex_idx]
        if len(region) > 5:
            threshold = region.max() * 0.20
            for k, val in enumerate(region):
                if val > threshold:
                    turnin_idx = entry_idx + k
                    turnin_method = 'g_lat_threshold'
                    break

    turnin_idx = max(entry_idx + 1, min(turnin_idx, apex_idx - 1))

    mc_start = int(np.argmin(np.abs(dist - (apex_dist - MC_RADIUS_M))))
    mc_end   = int(np.argmin(np.abs(dist - (apex_dist + MC_RADIUS_M))))
    mc_start = max(mc_start, turnin_idx)
    mc_end   = min(mc_end, exit_idx)

    slb_dist = float(dist[turnin_idx] - dist[entry_idx]) if turnin_idx > entry_idx else 0.0
    slb_speed_drop = float(speed[entry_idx] - speed[turnin_idx])
    slb_decel = slb_speed_drop / max(slb_dist, 1.0) * (1000.0 / 3600.0)

    ce_dist = float(dist[mc_start] - dist[turnin_idx]) if mc_start > turnin_idx else 0.0
    ce_brake_at_turnin = float(brake[turnin_idx]) if turnin_idx < len(brake) else 0.0

    mc_speeds = speed[mc_start:mc_end+1]
    mc_speed_ratio = float(mc_speeds.min() / mc_speeds.max()) if len(mc_speeds) > 1 and mc_speeds.max() > 0 else 1.0

    mc_lateral = 0.0
    if (mode == 'g_lat' or mode == 'steer') and steer_col in df_lap.columns:
        mc_lateral = float(np.abs(df_lap[steer_col].values[mc_start:mc_end+1]).mean())

    throttle_lag = 0.0
    for j in range(apex_idx, exit_idx):
        if acc[j] > 0.3:
            throttle_lag = float(dist[j] - dist[apex_idx])
            break

    cex_region = speed[apex_idx:exit_idx+1]
    if len(cex_region) > 1:
        cex_accel = float(cex_region[-1] - cex_region[0]) / max(float(dist[exit_idx] - dist[apex_idx]), 1.0)
    else:
        cex_accel = 0.0

    return {
        'phase_slb_dist': slb_dist,
        'phase_slb_decel_rate': slb_decel,
        'phase_ce_dist': ce_dist,
        'phase_ce_brake_at_turnin': ce_brake_at_turnin,
        'phase_mc_speed_ratio': mc_speed_ratio,
        'phase_mc_lateral_signal': mc_lateral,
        'phase_cex_throttle_lag': throttle_lag,
        'phase_cex_accel_rate': cex_accel,
        'phase_turnin_method': turnin_method,
    }


def detect_steer_column(df):
    cols = {c: c.lower() for c in df.columns}
    for c, cl in cols.items():
        if any(k in cl for k in ['steerangle', 'steering_angle', 'steer_angle', 'wheel_angle']):
            return ('steer', c)
    for c, cl in cols.items():
        if 'steer' in cl and 'error' not in cl:
            return ('steer', c)
    for c, cl in cols.items():
        if any(k in cl for k in ['g_lat', 'glat', 'lateral_g', 'accg_y', 'accel_lat']):
            return ('g_lat', c)
    return ('speed_proxy', None)


def wavg(values, weights):
    v    = pd.to_numeric(pd.Series(values), errors='coerce').values
    w    = np.array(weights, dtype=float)
    mask = ~np.isnan(v) & (w > 0)
    return float(np.average(v[mask], weights=w[mask])) if mask.sum() > 0 else np.nan

print("Core fonksiyonlar yuklendi (NB14 ile ayni)")

## 2. SAC Yardimcilari (adapter, lap split, temizlik filtresi, matrix builder)

In [ ]:
def adapt_sac(df):
    # SAC zaten ACGym-native (brakeStatus, accStatus 0-1, steerAngle, speed m/s, LapDist)
    df = df.copy()
    if 'speed_kmh' not in df.columns and 'speed' in df.columns:
        df['speed_kmh'] = pd.to_numeric(df['speed'], errors='coerce') * 3.6
    return df

def read_sac(f):
    try:
        return pd.read_parquet(f, columns=SAC_READ_COLS)
    except Exception:
        return pd.read_parquet(f)

def split_laps_sac(df):
    # LapDist resetlerinden turlara bol
    dist = pd.to_numeric(df['LapDist'], errors='coerce').values
    resets = list(np.where(np.diff(dist) < -1000)[0] + 1)
    bounds = [0] + resets + [len(df)]
    laps = [df.iloc[a:b].reset_index(drop=True) for a, b in zip(bounds[:-1], bounds[1:]) if b - a > 50]
    return laps if laps else [df]

def is_clean_lap(lap, min_span):
    dist = pd.to_numeric(lap['LapDist'], errors='coerce')
    if (dist.max() - dist.min()) < min_span:
        return False
    if 'speed_kmh' in lap.columns and pd.to_numeric(lap['speed_kmh'], errors='coerce').max() < MIN_LAP_MAXSPEED:
        return False
    for flag in ['going_backwards', 'isInPit']:
        if flag in lap.columns:
            if pd.to_numeric(lap[flag], errors='coerce').fillna(0).mean() > 0.05:
                return False
    return True

def build_matrix(corners_df, driver_id, track, n_laps):
    w = corners_df['confidence'].fillna(0).values
    valid = corners_df[corners_df['confidence'] > 0]
    has_cc = 'character_class' in valid.columns and len(valid)
    def pct(cls):
        return float((valid['character_class'] == cls).mean()) if has_cc else np.nan
    return {
        'driver_id': driver_id, 'track': track,
        'n_corners_total': len(corners_df), 'n_corners_valid': len(valid), 'n_laps': n_laps,
        'mean_apex_speed':     wavg(corners_df['seg_apex_speed'], w),
        'mean_entry_speed':    wavg(corners_df['seg_entry_speed'], w),
        'mean_exit_speed':     wavg(corners_df['seg_exit_speed'], w),
        'speed_loss_eff':      wavg(corners_df['seg_speed_loss_eff'], w),
        'mean_braking_dist':   wavg(corners_df['seg_braking_dist'], w),
        'mean_brake_pressure': wavg(corners_df['seg_avg_brake_pressure'], w),
        'mean_coasting_dist':  wavg(corners_df['seg_coasting_dist'], w),
        'trail_braking_ratio': float(valid['seg_trail_braking'].mean()) if len(valid) else np.nan,
        'mean_trail_pressure': wavg(corners_df['seg_trail_pressure'], w),
        'apex_speed_std':      float(valid['seg_apex_speed'].std()) if len(valid) else np.nan,
        'braking_dist_std':    float(valid['seg_braking_dist'].std()) if len(valid) else np.nan,
        'exit_speed_std':      float(valid['seg_exit_speed'].std()) if len(valid) else np.nan,
        'speed_loss_eff_std':  float(valid['seg_speed_loss_eff'].std()) if len(valid) else np.nan,
        'pct_heavy_braking':   pct('heavy_braking'),
        'pct_trail_braking':   pct('trail_braking'),
        'pct_lift_coast':      pct('lift_coast'),
        'pct_flat_out':        pct('flat_out'),
        'mean_slb_dist':         wavg(corners_df['phase_slb_dist'], w),
        'mean_slb_decel':        wavg(corners_df['phase_slb_decel_rate'], w),
        'mean_ce_dist':          wavg(corners_df['phase_ce_dist'], w),
        'mean_ce_brake_turnin':  wavg(corners_df['phase_ce_brake_at_turnin'], w),
        'mean_mc_speed_ratio':   wavg(corners_df['phase_mc_speed_ratio'], w),
        'mean_mc_lateral':       wavg(corners_df['phase_mc_lateral_signal'], w),
        'mean_cex_throttle_lag': wavg(corners_df['phase_cex_throttle_lag'], w),
        'mean_cex_accel_rate':   wavg(corners_df['phase_cex_accel_rate'], w),
    }

def segment_lap(df_lap, corners_v3, steer_info, lap_idx=0, car='SAC'):
    rows = []
    for _, crow in corners_v3.iterrows():
        apex_d = float(crow.get('apex_dist', crow.get('apex_lapdist', 0)))
        diff   = float(crow.get('difficulty_score', 0.5))
        seg = segment_corner(df_lap, apex_d, diff)
        seg.update(segment_corner_phases(df_lap, seg, steer_info))
        seg['corner_id'] = crow.get('corner_id', lap_idx)
        seg['n_votes']   = crow.get('n_votes', 3)
        seg['confidence'] = votes_to_confidence(seg.get('n_votes', 3))
        if 'character_class' in crow.index:
            seg['character_class'] = crow['character_class']
        seg['lap_idx'] = lap_idx; seg['car'] = car
        if not seg.get('seg_entry_valid', False):
            seg['confidence'] = 0.0
        if seg.get('seg_apex_speed', 0) < MIN_APEX_SPEED:
            seg['confidence'] = 0.0
        if seg.get('seg_entry_speed', 0) < seg.get('seg_apex_speed', 0):
            seg['confidence'] = 0.0
        rows.append(seg)
    return rows

print("SAC yardimci + build_matrix + segment_lap yuklendi")

## 3. SAC Referansini Kur (pist basina temiz turlar -> segment -> agregat)

In [ ]:
def resolve_track_files(token):
    toks = [token] + (['red_bull_ring', 'redbull', 'rbr'] if token == 'red_bull' else [])
    corners_f = next((h for t in toks for h in FEATURES_DIR.glob(f"*{t}*corners_v3*.parquet")), None)
    human_f   = next((h for t in toks for h in FEATURES_DIR.glob(f"driver_corner_matrix*{t}*.parquet")), None)
    return corners_f, human_f

def _clean(lap, min_span):
    # is_clean_lap'in hucre-ici, kendine-yeten surumu (imza bagimliligi yok)
    dist = pd.to_numeric(lap['LapDist'], errors='coerce')
    dmax = dist.max()
    if not np.isfinite(dmax) or (dmax - dist.min()) < min_span:
        return False
    if 'speed_kmh' in lap.columns and pd.to_numeric(lap['speed_kmh'], errors='coerce').max() < MIN_LAP_MAXSPEED:
        return False
    for fl in ['going_backwards', 'isInPit']:
        if fl in lap.columns and pd.to_numeric(lap[fl], errors='coerce').fillna(0).mean() > 0.05:
            return False
    return True

TRACK_TOKENS = ['monza', 'barcelona', 'red_bull']
sac_files_all = sorted(SAC_DIR.rglob("*.parquet"))
sac_rows = []
sac_corner_pool = {}

for token in TRACK_TOKENS:
    corners_f, human_f = resolve_track_files(token)
    if corners_f is None:
        print(f"[{token}] corners_v3 bulunamadi - atlandi"); continue
    corners_v3 = pd.read_parquet(corners_f)
    track_files = [f for f in sac_files_all if token in f.name.lower()]
    if not track_files:
        print(f"[{token}] SAC dosyasi yok - atlandi"); continue
    step = max(1, len(track_files) // (SAC_SAMPLE_N * 2))
    cand = track_files[::step]
    # v3.2: aralikli havuz tukenirse kalan dosyalara devam et (sira korunur;
    # havuz yeterliyse davranis birebir aynidir)
    _seen = set(cand)
    cand = cand + [f for f in track_files if f not in _seen]

    # v3.2 pist uzunlugu — KESTIRIM DEGIL, SABIT.
    # v3.1'e kadar `cand[:30]` uzerinden LapDist.max() MEDYANI aliniyordu.
    # SAC referans havuzu turdes degil: Barcelona dosyalari tam tur, Monza ve
    # Red Bull Ring dosyalarinin buyuk kismi erken sonlanan RL parcalari.
    # Medyan bu parcalarin icine dustu ->
    #     Monza     2142 m (gercek 5758) -> esik turun %30'u
    #     Red Bull  2910 m (gercek 4286) -> esik turun %54'u
    #     Barcelona 4592 m (gercek 4592) -> dogru
    # Monza'da viraj gecerliligi bu yuzden %85.3'e dusuyordu (digerleri %98-99.9).
    # Asagidaki degerler INSAN telemetrisinden olculdu (tier3 *_human.parquet,
    # LapDist span maksimumu) ve yayimlanmis pist uzunluklariyla uyumlu.
    TRACK_LEN_M = {'monza': 5757.9, 'barcelona': 4591.7, 'red_bull': 4286.1}
    MIN_SPAN_FRAC = 0.95        # v3.1'de 0.80; tam tur bollugu var (bkz. asagidaki sayim)
    track_len = TRACK_LEN_M.get(token)
    assert track_len is not None, f"{token} icin dogrulanmis pist uzunlugu yok"
    min_span = MIN_SPAN_FRAC * track_len
    print(f"[{token}] track_len={track_len:.1f}m (sabit) min_span={min_span:.0f}m "
          f"(turun %{MIN_SPAN_FRAC*100:.0f}'i)")

    all_corners, clean_laps, steer_info, skipped = [], 0, None, 0
    for f in cand:
        if clean_laps >= SAC_SAMPLE_N:
            break
        try:
            df = adapt_sac(read_sac(f))
        except Exception:
            continue
        for lap in split_laps_sac(df):
            if clean_laps >= SAC_SAMPLE_N:
                break
            try:
                if not _clean(lap, min_span):
                    continue
                if steer_info is None:
                    steer_info = detect_steer_column(lap)
                all_corners.extend(segment_lap(lap, corners_v3, steer_info, lap_idx=clean_laps))
                clean_laps += 1
            except Exception as e:
                skipped += 1
                continue

    if not all_corners:
        print(f"[{token}] temiz tur bulunamadi - atlandi (atlanan_hatali_tur={skipped})"); continue
    cdf = pd.DataFrame(all_corners)
    sac_corner_pool[token] = cdf
    m = build_matrix(cdf, driver_id=f'SAC_{token}', track=token, n_laps=clean_laps)
    m['_corners_file'] = corners_f.name
    m['_human_file'] = human_f.name if human_f else 'YOK'
    sac_rows.append(m)
    vt = int((cdf['confidence'] > 0).sum())
    # v3: paydalar ve sayaclar JSON'a — 'gecerli viraj 1407/1650' IKI sayidir
    m['_n_corners_attempted'] = int(len(cdf))
    m['_clean_laps'] = int(clean_laps)
    m['_skipped_laps'] = int(skipped)
    m['_steer_col'] = str(steer_info)
    m['_track_len_m'] = float(track_len)
    m['_track_len_source'] = 'sabit (insan telemetrisinden olculdu, v3.2)'
    m['_min_span_m'] = float(min_span)
    m['_min_span_frac'] = float(MIN_SPAN_FRAC)
    m['_n_files_scanned'] = int(len(cand))
    print(f"[{token}] steer={steer_info} temiz_tur={clean_laps} gecerli_viraj={vt}/{len(cdf)} atlanan={skipped} corners={corners_f.name} human={m['_human_file']}")

sac_matrix = pd.DataFrame(sac_rows)
print(f"\nSAC referans matrisi: {len(sac_matrix)} pist")

## 4. SAC Referans Matrisini Kaydet

In [ ]:
keep = ['driver_id','track','n_laps','n_corners_valid','_corners_file','_human_file'] + ALL_METRICS
cols = [c for c in keep if c in sac_matrix.columns]
sac_matrix[cols].to_parquet(OUT_DIR / "sac_reference_matrix.parquet", index=False)
print("Kaydedildi: sac_reference_matrix.parquet")
print(sac_matrix[['track','n_laps','n_corners_valid','mean_apex_speed','mean_brake_pressure','trail_braking_ratio','mean_coasting_dist']].to_string(index=False))

## 5. Degenerate Metrik Tespiti (RATIO vs MINMAX)

In [ ]:
# Sinifllama kurali: bir metrik RATIO (insan/SAC) olur ANCAK eger
#   (1) B4 tutarlilik (std) DEGIL  -> std populasyon-goreli, oran anlamsiz
#   (2) pct_ DEGIL                 -> oran-of-oran
#   (3) insan populasyonunda negatif deger YOK -> oran isareti anlamli kalir
#   (4) SAC referansi |.| >= RATIO_EPS -> payda ~0 degil
# aksi halde MINMAX (populasyon Min-Max, mevcut HNS-disi karar)
ref_human = None
_mh = next((r['_human_file'] for r in sac_rows if r['track'] == 'monza' and r.get('_human_file','YOK') != 'YOK'), None)
if _mh:
    ref_human = read_human(FEATURES_DIR / _mh)

def classify_metric(met):
    if met in DIMENSIONS['B4_Tutarlilik']['metrics']:
        return 'MINMAX'
    if met.startswith('pct_'):
        return 'MINMAX'
    if ref_human is not None and met in ref_human.columns:
        col = pd.to_numeric(ref_human[met], errors='coerce')
        if pd.notna(col.min()) and col.min() < 0:
            return 'MINMAX'
    vals = [r.get(met, np.nan) for r in sac_rows]
    vals = [v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
    ref = float(np.nanmedian(vals)) if vals else np.nan
    if (not vals) or abs(ref) < RATIO_EPS:
        return 'MINMAX'
    return 'RATIO'

print("=== METRIK SINIFLAMA (RATIO vs MINMAX) ===")
metric_class = {}
for m in ALL_METRICS:
    vals = [r.get(m, np.nan) for r in sac_rows]
    vals = [v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
    ref = float(np.nanmedian(vals)) if vals else np.nan
    metric_class[m] = classify_metric(m)
    dimn = next((d for d, c in DIMENSIONS.items() if m in c['metrics']), '?')
    neg = ''
    if ref_human is not None and m in ref_human.columns:
        cmin = pd.to_numeric(ref_human[m], errors='coerce').min()
        neg = ' [neg]' if (pd.notna(cmin) and cmin < 0) else ''
    rr = f"{ref:9.4f}" if vals else "    NaN"
    print(f"  {dimn:14s} {m:24s} SAC_ref_med={rr}{neg}  -> {metric_class[m]}")
ratio_metrics  = [m for m, k in metric_class.items() if k == 'RATIO']
minmax_metrics = [m for m, k in metric_class.items() if k == 'MINMAX']
print(f"\nRATIO (SAC-oran): {len(ratio_metrics)} | MINMAX: {len(minmax_metrics)}")
print(f"  RATIO  : {ratio_metrics}")
print(f"  MINMAX : {minmax_metrics}")

## 6. SAC-ratio Fingerprint (hibrit: oran + degenerate icin MinMax)

In [ ]:
def build_sac_ratio_fingerprint(human, sac_ref):
    """v3.1: indeks hizalama korumasi + dejenere MINMAX metrik kaydi.

    v3'te iki sessiz sorun vardi:
      (a) `human` bosluklu indeksle geliyordu; `norm` temiz RangeIndex ile
          kuruldugu icin indeks-hizali atama satir dusuruyordu.
      (b) sabit (mx == mn) bir MINMAX metrigi butun satirlara 0.5 yaziyor ve
          bu, o boyutun `n_valid` sayisini yapay olarak dolduruyordu.
    Ikisi de artik olculuyor ve raporlaniyor.
    """
    human = human.reset_index(drop=True)   # savunma katmani (read_human da yapiyor)
    _degen, _align_loss = [], {}
    # 1) metrik bazli normalize tablo (RATIO -> human/SAC, MINMAX -> populasyon min-max)
    norm = pd.DataFrame({'driver_id': human['driver_id'].values})
    for met in ALL_METRICS:
        if met not in human.columns:
            norm[met] = np.nan; continue
        col = pd.to_numeric(human[met], errors='coerce')
        n_src = int(col.notna().sum())
        if metric_class.get(met) == 'RATIO':
            sv = sac_ref.get(met, np.nan)
            _ok = bool(sv) and abs(sv) > RATIO_EPS
            norm[met] = (col / sv) if _ok else np.nan
            n_exp = n_src if _ok else 0
        else:
            mn, mx = col.min(), col.max()
            if mx > mn:
                norm[met] = (col - mn) / (mx - mn); n_exp = n_src
            else:
                norm[met] = 0.5; n_exp = len(norm); _degen.append(met)
        n_got = int(norm[met].notna().sum())
        if n_got != n_exp:
            _align_loss[met] = {'beklenen': n_exp, 'gelen': n_got}
    assert not _align_loss, (
        "INDEKS HIZALAMA KAYBI - `human` reset_index'siz geldi.\n" + str(_align_loss))
    if _degen:
        print(f"    [uyari] sabit MINMAX metrigi 0.5 ile dolduruldu: {_degen}")
    # 2) boyut skoru
    fp = pd.DataFrame({'driver_id': human['driver_id'].values})
    for dim, cfg in DIMENSIONS.items():
        mets = [m for m in cfg['metrics'] if m in norm.columns]
        sub = norm[mets].apply(pd.to_numeric, errors='coerce')
        score = sub.mean(axis=1, skipna=True)
        if dim == 'B4_Tutarlilik':
            score = 1.0 - score   # std tabanli: yuksek = dusuk tutarlilik
        fp[dim] = score.values
    return fp, norm, _degen

_FP_STATS = {}
for r in sac_rows:
    token = r['track']
    if r.get('_human_file', 'YOK') == 'YOK':
        print(f"[{token}] human matrix yok - SAC-ratio fingerprint atlandi"); continue
    human = read_human(FEATURES_DIR / r['_human_file'])
    fp, norm, _degen = build_sac_ratio_fingerprint(human, r)
    fp.to_parquet(OUT_DIR / f"sac_ratio_fingerprint_{token}.parquet", index=False)
    print(f"[{token}] SAC-ratio fingerprint: {len(fp)} surucu -> sac_ratio_fingerprint_{token}.parquet")
    # v3: politikanin ETKILEDIGI sayilar — pist basina SAC-oran dagilimlari
    _FP_STATS[token] = {"n_drivers": int(len(fp)),
                        "minmax_degenerate": list(_degen), "dims": {}}
    for dim in DIM_NAMES:
        _FP_STATS[token]["dims"][dim] = {
            "mean": float(fp[dim].mean()), "min": float(fp[dim].min()),
            "max": float(fp[dim].max()), "std": float(fp[dim].std()),
            "n_valid": int(fp[dim].notna().sum())}
        _nv, _nd = int(fp[dim].notna().sum()), int(len(fp))
        _bayrak = "" if _nv == _nd else f"   <-- {_nd - _nv} surucu NaN"
        print(f"    {dim:16s} ort={fp[dim].mean():.3f}  "
              f"(min={fp[dim].min():.3f} max={fp[dim].max():.3f})  n={_nv}/{_nd}{_bayrak}")

## 7. Efe Monza SAC-Gap

In [ ]:
efe_hits = [f for f in FEATURES_DIR.glob("*.parquet") if 'efe' in f.name.lower()]
sac_monza = next((r for r in sac_rows if r['track'] == 'monza'), None)
if not efe_hits:
    print("Efe matrix bulunamadi (data/features icinde *efe*). NB14 ciktisini kontrol et.")
elif sac_monza is None:
    print("Monza SAC referansi yok - Efe gap atlandi.")
else:
    efe_row, efe_src = None, None
    for f in efe_hits:
        d = pd.read_parquet(f)
        if 'driver_id' in d.columns and (d['driver_id'] == 'EFE_USER').any():
            efe_row = d[d['driver_id'] == 'EFE_USER'].iloc[0]; efe_src = f.name; break
        if len(d) == 1 and any(m in d.columns for m in ALL_METRICS):
            efe_row = d.iloc[0]; efe_src = f.name; break
    if efe_row is None:
        print(f"Efe metrik satiri ayirt edilemedi. Aday dosyalar: {[f.name for f in efe_hits]}")
    else:
        print(f"Efe kaynak: {efe_src}")
        print("\n=== EFE vs SAC (Monza) - SAC-ratio gap (1.0 = SAC ile esit) ===")
        gap_rows = []
        for dim, cfg in DIMENSIONS.items():
            comps = []
            for met in cfg['metrics']:
                if metric_class.get(met) != 'RATIO':
                    continue
                if met not in efe_row.index or pd.isna(efe_row[met]):
                    continue
                sv = sac_monza.get(met, np.nan)
                if sv and abs(sv) > RATIO_EPS:
                    comps.append(float(efe_row[met]) / sv)
            score = float(np.nanmean(comps)) if comps else np.nan
            tag = ('SAC-ustu' if score > 1 else 'SAC-alti') if comps else 'sadece-MinMax-boyut'
            gap_rows.append({'dimension': dim, 'sac_ratio': score, 'n_ratio_metrics': len(comps)})
            sv_str = f"{score:.3f}" if comps else "  NaN"
            print(f"  {dim:16s} SAC-ratio={sv_str}  ({len(comps)} ratio-metrik)  [{tag}]")
        pd.DataFrame(gap_rows).to_parquet(OUT_DIR / "efe_sac_gap_monza.parquet", index=False)
        print(f"\nKaydedildi: efe_sac_gap_monza.parquet")

## 8. Ciktilar

In [ ]:
print("=== results/gap_analysis ciktilari ===")
for f in sorted(OUT_DIR.glob("*.parquet")):
    print(f"  {f.name:38s} {f.stat().st_size/1024:7.1f} KB")
print("\nNB15 tamamlandi.")

In [ ]:
from pathlib import Path
import pandas as pd

P = (TRACK_ROOT / "data" / "processed" / "tier3_large_3t3c")
TRACKS = ['monza', 'barcelona', 'red_bull_ring']
CARS   = ['bmw_z4_gt3', 'dallara_f317', 'ks_mazda_miata']
DEG    = EXCLUDE_IDS   # v3: ortak politika

rows = []
for t in TRACKS:
    for c in CARS:
        f = P / f"{t}_{c}_human.parquet"
        if not f.exists():
            continue
        df = pd.read_parquet(f, columns=['driver_id'])
        for d in df['driver_id'].unique():
            if d not in DEG:
                rows.append({'track': t, 'car': c, 'driver_id': d})

dc  = pd.DataFrame(rows)
piv = dc.pivot_table(index='driver_id', columns='car',
                     values='track', aggfunc='count', fill_value=0)
piv['n_cars'] = (piv > 0).sum(axis=1)

print(piv.sort_values('n_cars', ascending=False).to_string())
print("\nAraç sayısına göre sürücü dağılımı:")
print(piv['n_cars'].value_counts().sort_index())

## 9. Izlenebilirlik kaydi (v3) — K4'un tek kaynagi


In [ ]:
import json as _json
from datetime import datetime as _dt
_gap    = {r['dimension']: (None if pd.isna(r['sac_ratio']) else float(r['sac_ratio'])) for r in gap_rows} if gap_rows else None
_nratio = {r['dimension']: int(r['n_ratio_metrics']) for r in gap_rows} if gap_rows else None
_cls    = {m: classify_metric(m) for d in DIMENSIONS.values() for m in d['metrics']}
_info = {
    'notebook': 'NB15_sac_gap_analysis_v3_2',
    'amendment': ('v3.1: read_human reset_index eklendi. v3te filtrelenmis cerceve '
                  'bosluklu indeksle geliyordu, indeks-hizali atama sac_ratio_fingerprint'
                  'ten satir dusuruyordu (Monza EXC 14 surucu -> 11 gecerli). '
                  'efe_gap_monza ETKILENMEDI (yalniz RATIO metrikleri, populasyondan bagimsiz).'
                  ' | v3.2: pist uzunlugu kestirimi KALDIRILDI, dogrulanmis sabit degerlere '
                  'gecildi (Monza 2142->5757.9, RBR 2910->4286.1); temiz-tur esigi 0.80->0.95. '
                  'v3.1 ve oncesinde Monza referansi kismi turlarla kirlenmisti.'),
    'run_timestamp': _dt.now().isoformat(timespec='seconds'),
    'identity_policy': POLICY,
    'ablock_excluded': (POLICY == 'P_EXC'),
    'exclude_ids': EXCLUDE_IDS,
    'human_filter_log': _HUMAN_LOG,
    'sac_reference': [{'track': r['track'], 'n_laps': int(r['n_laps']),
                       'n_corners_valid': int(r['n_corners_valid']),
                       'human_file': r.get('_human_file', 'YOK'),
                       'mean_apex_speed': float(r['mean_apex_speed']),
                       'mean_brake_pressure': float(r['mean_brake_pressure']),
                       'trail_braking_ratio': float(r['trail_braking_ratio']),
                       'mean_coasting_dist': float(r['mean_coasting_dist']),
                       'n_corners_attempted': int(r['_n_corners_attempted']),
                       'clean_laps': int(r['_clean_laps']),
                       'skipped_laps': int(r['_skipped_laps']),
                       'steer_col': r['_steer_col'],
                       'track_len_m': float(r['_track_len_m']),
                       'track_len_source': r.get('_track_len_source'),
                       'min_span_m': float(r.get('_min_span_m', float('nan'))),
                       'min_span_frac': float(r.get('_min_span_frac', float('nan'))),
                       'n_files_scanned': int(r.get('_n_files_scanned', -1))} for r in sac_rows],
    'metric_classification': _cls,
    'n_ratio':  sum(1 for v in _cls.values() if v == 'RATIO'),
    'n_minmax': sum(1 for v in _cls.values() if v == 'MINMAX'),
    'sac_sample_n': SAC_SAMPLE_N,
    'n_sac_parquet': N_SAC_PARQUET,
    'sac_ratio_fingerprint': _FP_STATS,
    'fingerprint_integrity': {t: {'n_drivers': v['n_drivers'],
                                  'n_valid_per_dim': {d: v['dims'][d]['n_valid'] for d in DIM_NAMES},
                                  'minmax_degenerate': v.get('minmax_degenerate', [])}
                              for t, v in _FP_STATS.items()},
    'efe_gap_monza': _gap,
    'efe_gap_n_ratio_metrics': _nratio,
}
with open(OUT_DIR / 'nb15_info.json', 'w', encoding='utf-8') as _f:
    _json.dump(_info, _f, indent=2, ensure_ascii=False)
print(f"Izlenebilirlik yazildi: {OUT_DIR / 'nb15_info.json'}")
print(_json.dumps({k: v for k, v in _info.items() if k != 'metric_classification'}, indent=1, ensure_ascii=False)[:1200])


In [ ]:
import json, glob, os

for p in sorted(glob.glob(str(TRACK_ROOT / "results" / "gap_analysis_*" / "nb15_info.json"))):
    d = json.load(open(p, encoding='utf-8'))
    print('=' * 74)
    print(os.path.basename(os.path.dirname(p)), '|', d.get('notebook'), '|', d.get('run_timestamp'))
    print('--- sac_reference ---')
    for r in d.get('sac_reference', []):
        print('  {:<11} tur={:<4} viraj={}/{}  len={} ({})  esik={} [{}]  taranan={}'.format(
            r['track'], r['n_laps'], r['n_corners_valid'], r['n_corners_attempted'],
            round(r['track_len_m'], 1), r.get('track_len_source', '?'),
            round(r.get('min_span_m', float('nan')), 0), r.get('min_span_frac'),
            r.get('n_files_scanned')))
        print('     apex={:.2f}  brake={:.4f}  trail={:.4f}  coast={:.3f}'.format(
            r['mean_apex_speed'], r['mean_brake_pressure'],
            r['trail_braking_ratio'], r['mean_coasting_dist']))
    print('--- efe_gap_monza ---', json.dumps(d.get('efe_gap_monza'), ensure_ascii=False))
    print('--- fingerprint_integrity ---')
    print(json.dumps(d.get('fingerprint_integrity'), indent=1, ensure_ascii=False))
    print('--- sac_ratio_fingerprint (ozet) ---')
    for t, v in (d.get('sac_ratio_fingerprint') or {}).items():
        print('  {:<11} n={}'.format(t, v['n_drivers']),
              {k: round(x['mean'], 4) for k, x in v['dims'].items()})

In [ ]:
import json, glob, os
for p in sorted(glob.glob(str(TRACK_ROOT / "results" / "cross_car_*" / "nb16_info.json"))):
    d = json.load(open(p, encoding='utf-8'))
    print('=' * 74); print(os.path.basename(os.path.dirname(p)))
    for k in ['ladder','design','rung2','rung3_bh_table','identifiability_summary',
              'identifiability_pooled','identifiability_within_track',
              'dimension_identifiability','repeatability','calibration',
              's1_dimensionless_vs_dimensional','s2_derived','sensitivity',
              'leave_one_metric_out','verdict_code','verdict_text']:
        print('---', k)
        print(json.dumps(d.get(k), indent=1, ensure_ascii=False))